# Notebook 02 — Classification Analysis

Behavioural feature exploration + genotype classification. Pool multiple tracking runs (same day / genotype layout) for more flies per genotype.

## Stages
1. Configuration (`RUN_DIRS`, `CV_MODE`, auto-incremented `FIGURES_DIR`)
2. Load `compact_tracks.csv` per run, map genotypes, concat with run-scoped `compact_id`
3. Extract frame-level behavioural features
4. Aggregate to per-fly features (+ `run` column)
5. Feature list + **per-run sanity** box plots (catch video/batch offsets)
6. Exploratory plots (by genotype, WT vs mutant)
7. Classification (LDA, Logistic, SVC): **stratified** CV or **group** CV (leave-one-video-out)
8. Optional: single `24DPE.html` — pooled + per-trial report (same plots + classifiers unpooled)

**Edit the configuration cell** (`run_multiple_videos`, `RUN_DIRS` / tag, `CV_MODE`).

In [1]:
import sys
sys.path.insert(0, '..')  # so 'src' is importable from notebooks/

import os
import pandas as pd
import plotly.express as px

from src.classification import (
    map_vial_to_genotype,
    run_classifier,
    plot_by_genotype,
    plot_wt_vs_mutant,
    write_classification_html_report,
)
from src.features import extract_behavioral_features, aggregate_per_fly_features

## 1 — Configuration

In [2]:
# ---- EDIT THIS ----
run_multiple_videos = True
CV_MODE = "group"  # "stratified" — pooled flies | "group" — leave-one-video-out (GroupKFold)
assert CV_MODE in ("stratified", "group")

if run_multiple_videos:
    RUN_DIRS = [f"../outputs/run_{n}_24DPE_n{n - 96:03d}" for n in range(97, 103)]
    tag = "24DPE_n001_to_n006"
else:
    RUN_DIRS = [r"../outputs/run_64_41DPE_n004"]
    tag = os.path.basename(os.path.normpath(RUN_DIRS[0]))

classif_root = "../outputs/classification"
os.makedirs(classif_root, exist_ok=True)
prefix = f"{tag}_run"
existing = [d for d in os.listdir(classif_root) if d.startswith(prefix)]
nums = []
for d in existing:
    rest = d.removeprefix(prefix)
    if rest.isdigit():
        nums.append(int(rest))
FIGURES_DIR = os.path.join(classif_root, f"{tag}_run{max(nums, default=0) + 1}")
os.makedirs(FIGURES_DIR, exist_ok=True)
print("Figures dir:", FIGURES_DIR)

Figures dir: ../outputs/classification\24DPE_n001_to_n006_run7


## 2 — Load data and map genotypes

`map_vial_to_genotype` parses the filename to infer which vial corresponds
to which genotype (e.g. `..._hTDP43_WT-Het-Homo_...`).

In [3]:
parts = []
for rd in RUN_DIRS:
    d = map_vial_to_genotype(rd)
    run_tag = os.path.basename(os.path.normpath(rd))
    d["run"] = run_tag
    d["compact_id"] = run_tag + "::" + d["compact_id"].astype(str)
    parts.append(d)
df_raw = pd.concat(parts, ignore_index=True)
print(df_raw.shape, "| runs:", df_raw["run"].nunique())
print(df_raw["genotype"].value_counts())
df_raw.head()

(57263, 10) | runs: 6
genotype
M337V    10777
A315T    10296
A90V      9783
G294A     9560
G287S     9244
WT        7603
Name: count, dtype: int64


,frame,orig_id,x,y,stitched_id,vial_id,compact_id,fps,genotype,run
0,0,id1,298.0,303.5,id1,vial3,run_97_24DPE_n001::15,30.0,G287S,run_97_24DPE_n001
1,1,id1,295.0,302.5,id1,vial3,run_97_24DPE_n001::15,30.0,G287S,run_97_24DPE_n001
2,2,id1,292.5,302.0,id1,vial3,run_97_24DPE_n001::15,30.0,G287S,run_97_24DPE_n001
3,3,id1,289.5,301.5,id1,vial3,run_97_24DPE_n001::15,30.0,G287S,run_97_24DPE_n001
4,4,id1,287.5,300.5,id1,vial3,run_97_24DPE_n001::15,30.0,G287S,run_97_24DPE_n001


## 3 — Extract behavioural features

Computes frame-level kinematics (velocity, acceleration, turning angle),
convex-hull area, and path tortuosity for each fly.

In [4]:
df_feat = extract_behavioral_features(df_raw)
print(df_feat.shape)
df_feat[["compact_id", "frame", "velocity", "turning_angle", "area_covered", "tortuosity"]].head()

(57247, 22)


,compact_id,frame,velocity,turning_angle,area_covered,tortuosity
0,run_100_24DPE_n004::1,0,0.000000,0.000000,15005.125,1.38461
1,run_100_24DPE_n004::1,1,15.000000,1.570796,15005.125,1.38461
2,run_100_24DPE_n004::1,2,30.000000,1.570796,15005.125,1.38461
3,run_100_24DPE_n004::1,5,25.495098,0.197396,15005.125,1.38461
4,run_100_24DPE_n004::1,6,33.541020,0.909753,15005.125,1.38461


## 4 — Aggregate to per-fly features

In [5]:
df_agg = aggregate_per_fly_features(df_feat, pause_threshold=1.0)

meta = (
    df_raw.drop_duplicates("compact_id")
    .set_index("compact_id")[["genotype", "run"]]
)
df_agg = df_agg.join(meta, on="compact_id").dropna(subset=["genotype"])

print(df_agg.shape)
df_agg.head()

(269, 13)


,compact_id,mean_velocity,median_velocity,std_velocity,pause_fraction,mean_abs_turning_angle,mean_abs_angular_velocity,total_distance_traveled,tortuosity,area_covered,vial_id,genotype,run
0,run_100_24DPE_n004::1,71.397085,60.000000,123.352397,0.148571,0.533116,15.882534,452.075275,1.384610,15005.125,vial1,WT,run_100_24DPE_n004
1,run_100_24DPE_n004::10,66.980267,43.713203,133.633004,0.104167,0.687196,20.583184,789.967177,2.425610,17874.000,vial2,A90V,run_100_24DPE_n004
2,run_100_24DPE_n004::11,77.949126,42.426407,156.433362,0.128743,0.776668,23.289154,887.984593,2.699440,17232.500,vial2,A90V,run_100_24DPE_n004
3,run_100_24DPE_n004::12,24.966007,15.000000,70.525076,0.451327,0.891657,26.743512,325.605653,89.450752,1947.375,vial2,A90V,run_100_24DPE_n004
4,run_100_24DPE_n004::13,160.444409,63.639610,248.514566,0.200000,1.299331,38.979940,26.740735,1.111787,35.500,vial2,A90V,run_100_24DPE_n004


In [6]:
FEATURES = [
    "mean_velocity",
    "median_velocity",
    "pause_fraction",
    "total_distance_traveled",
    "tortuosity",
    "area_covered",
]

FEATURE_TITLES = {
    "mean_velocity":            "Mean velocity (px/s)",
    "median_velocity":          "Median velocity (px/s)",
    "pause_fraction":           "Pause fraction",
    "total_distance_traveled":  "Total distance traveled (px)",
    "tortuosity":               "Path tortuosity",
    "area_covered":             "Area covered (px^2)",
}

hover_data = ["compact_id", "run"]

In [7]:
# Per-run sanity check: batch / lighting effects should not dwarf genotype
for feat in FEATURES:
    fig = px.box(
        df_agg, x="run", y=feat, color="genotype",
        points="all", hover_data=hover_data,
        title=f"{FEATURE_TITLES[feat]} — distribution per run",
    )
    fig.update_traces(jitter=0.3, marker=dict(size=7, opacity=0.75))
    fig.write_html(os.path.join(FIGURES_DIR, f"{feat}_per_run.html"))
    fig.show()

## 5 — Exploratory visualisation

Box plots for each feature, grouped by genotype (uses `FEATURES` / `hover_data` from above).

In [8]:
plot_by_genotype(df_agg, FEATURES, FEATURE_TITLES, hover_data, outdir=FIGURES_DIR)

In [9]:
plot_wt_vs_mutant(df_agg, FEATURES, FEATURE_TITLES, hover_data, outdir=FIGURES_DIR)

## 6 — Classification

Train LDA, Logistic Regression, and SVC classifiers.  
`CV_MODE == "group"` uses **GroupKFold** (no fly from a held-out video in training). Needs **≥2 runs**.  
`CV_MODE == "stratified"` pools all flies across videos (default sklearn splitter).

Figures go to `FIGURES_DIR` (auto-incremented under `outputs/classification/`).

In [10]:
groups = df_agg["run"].values if CV_MODE == "group" else None
print(
    f"CV scheme: {CV_MODE}"
    + (f" ({df_agg['run'].nunique()} video groups, GroupKFold)" if groups is not None else " (stratified)"),
)

for model_name in ["lda", "logistic", "svc"]:
    for mode in ["multiclass", "binary"]:
        print(f"\n=== {model_name.upper()} [{mode}] ===")
        run_classifier(
            df=df_agg,
            outdir=FIGURES_DIR,
            model_name=model_name,
            classification_mode=mode,
            cv=5,
            plot_importance=True,
            groups=groups,
        )

CV scheme: group (6 video groups, GroupKFold)

=== LDA [multiclass] ===



=== LDA [binary] ===



=== LOGISTIC [multiclass] ===



=== LOGISTIC [binary] ===



=== SVC [multiclass] ===



=== SVC [binary] ===


## 7 — Combined HTML report (`24DPE.html`)

Single page: **pooled** (same as above) plus **each trial** (`run`) separately — same genotype / WT-vs-mutant plots and classifiers, unpooled. Open the file in a browser (Plotly from CDN).

In [ ]:
REPORT_HTML = os.path.join(classif_root, "24DPE.html")
_groups_report = df_agg["run"].values if CV_MODE == "group" else None
write_classification_html_report(
    df_agg,
    FEATURES,
    FEATURE_TITLES,
    hover_data,
    REPORT_HTML,
    trial_column="run",
    report_title=f"{tag} — pooled and per-trial",
    pooled_cv=5,
    pooled_cv_groups=_groups_report,
    per_trial_cv=5,
)
print("Wrote", os.path.abspath(REPORT_HTML))

## Summary

Per-figure exports (HTML + PNG) are under `FIGURES_DIR`. The combined report is `24DPE.html` next to those runs under `classif_root` (from §7). Open any HTML in a browser for interactive Plotly charts.